# GeneTropica Phase 14 — Molecular Dynamics Simulation

**Three-drug mechanism comparison on DENV NS5 RdRp (PDB 5CCV, Chain A)**

| Drug | Consensus Rank | Mechanism | Purpose |
|------|---------------|-----------|----------|
| Celecoxib | #1 (0.6846) | COX-2 inhibitor | Pipeline's top prediction |
| Methotrexate | #3 (0.4930) | DHFR / host-directed | Tests indirect mechanism |
| Dasabuvir | #17 (0.3758) | Non-nucleoside RdRp | Direct polymerase binder |

**Protocol:** 50 ns all-atom MD per drug using GROMACS + ACPYPE (GAFF2)

**Runtime estimate:** 3–9 days total (T4 GPU)

---
Russell Young — British School Jakarta

In [ ]:
# ============================================================
# Cell 1: Install dependencies (~15-20 min first run, includes
#          building GROMACS from source with CUDA GPU support)
# ============================================================
import os, subprocess, sys

# --- Step 1: Install Miniforge (provides mamba/conda for AmberTools) ---
MF = '/opt/miniforge'
if not os.path.exists(f'{MF}/bin/mamba'):
    print('Installing Miniforge...')
    !wget -qO /tmp/mf.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    !bash /tmp/mf.sh -b -p {MF} > /dev/null 2>&1
    print('  Miniforge installed')
else:
    print('Miniforge: already installed')

os.environ['PATH'] = f'{MF}/bin:' + os.environ['PATH']

# --- Step 2: Install AmberTools via mamba ---
# ACPYPE needs antechamber + parmchk2 + tleap (CLI tools from AmberTools)
r = subprocess.run('which antechamber', shell=True, capture_output=True)
if r.returncode != 0:
    print('Installing AmberTools (~3-5 min)...')
    !mamba install -c conda-forge ambertools -y -q 2>&1 | tail -5
else:
    print('AmberTools: already installed')

# --- Step 3: Build GROMACS from source with CUDA GPU support ---
# Both apt-get and conda-forge GROMACS are CPU-only on Colab.
# Building from source with -DGMX_GPU=CUDA enables H100/T4 acceleration.
GMX_PREFIX = '/opt/gromacs'
GMX_VER = '2024.4'

# Check if GPU-enabled GROMACS is already installed
gmx_ready = False
r = subprocess.run(f'{GMX_PREFIX}/bin/gmx --version 2>&1',
                   shell=True, capture_output=True, text=True)
if r.returncode == 0 and 'CUDA' in (r.stdout + r.stderr).upper():
    gmx_ready = True
    print('GROMACS (GPU/CUDA): already installed')

if not gmx_ready:
    print(f'Building GROMACS {GMX_VER} with CUDA GPU support (~10-15 min)...')

    # Remove any CPU-only GROMACS to avoid conflicts
    subprocess.run('mamba remove -y gromacs 2>/dev/null',
                   shell=True, capture_output=True)

    # Install build dependencies
    !apt-get update -qq && apt-get install -y -qq cmake build-essential > /dev/null 2>&1

    # Download GROMACS source
    print('  Downloading source...')
    !wget -qO /tmp/gromacs.tar.gz https://ftp.gromacs.org/gromacs/gromacs-{GMX_VER}.tar.gz
    !cd /tmp && tar xzf gromacs.tar.gz

    # Configure with CUDA GPU support
    print('  Configuring (cmake)...')
    !mkdir -p /tmp/gromacs-{GMX_VER}/build
    !cd /tmp/gromacs-{GMX_VER}/build && cmake .. \
        -DGMX_GPU=CUDA \
        -DGMX_BUILD_OWN_FFTW=ON \
        -DCMAKE_INSTALL_PREFIX={GMX_PREFIX} \
        -DREGRESSIONTEST_DOWNLOAD=OFF \
        2>&1 | tail -5

    # Build using all available cores
    print('  Compiling (this takes ~10 min)...')
    !cd /tmp/gromacs-{GMX_VER}/build && make -j$(nproc) 2>&1 | tail -3

    # Install
    !cd /tmp/gromacs-{GMX_VER}/build && make install > /dev/null 2>&1

    # Cleanup source to save disk space
    !rm -rf /tmp/gromacs*

    print(f'  GROMACS {GMX_VER} with CUDA: installed to {GMX_PREFIX}')

# Put GPU GROMACS first on PATH (before any conda gmx)
os.environ['PATH'] = f'{GMX_PREFIX}/bin:' + os.environ['PATH']

# --- Step 4: Python packages via pip ---
print('Installing Python packages...')
!pip install -q acpype MDAnalysis matplotlib numpy 2>&1 | tail -3

# --- Verify all tools ---
print('\n--- Verification ---')
!{GMX_PREFIX}/bin/gmx --version 2>&1 | head -5
print()

# Check GPU/CUDA support
gpu_info = subprocess.run(f'{GMX_PREFIX}/bin/gmx --version 2>&1',
                          shell=True, capture_output=True, text=True)
gpu_out = gpu_info.stdout + gpu_info.stderr
if 'CUDA' in gpu_out.upper():
    print('  GROMACS GPU support: CUDA enabled ✓')
else:
    print('  WARNING: GROMACS GPU support not detected!')

# Check nvidia-smi
nvidia = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                        shell=True, capture_output=True, text=True)
if nvidia.returncode == 0:
    print(f'  GPU device: {nvidia.stdout.strip()} ✓')
else:
    print('  WARNING: No GPU device found!')

print()
for tool in ['antechamber', 'tleap', 'parmchk2']:
    r = subprocess.run(f'which {tool}', shell=True,
                       capture_output=True, text=True)
    status = f'OK ({r.stdout.strip()})' if r.returncode == 0 else 'NOT FOUND!'
    print(f'  {tool}: {status}')

for pkg in ['acpype', 'MDAnalysis', 'matplotlib', 'numpy']:
    try:
        __import__(pkg)
        print(f'  {pkg}: OK')
    except ImportError:
        print(f'  {pkg}: NOT FOUND!')

print('\n=== Installation complete ===')

In [ ]:
# ============================================================
# Cell 2: Upload input files (select ALL 8 files)
# ============================================================
from google.colab import files
import os

WORKDIR = '/content/md_simulation'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print('Please upload ALL 8 files from data/md_simulation/:')
print('  From celecoxib/input/  : protein_5CCV.pdb, celecoxib_docked.mol2')
print('  From methotrexate/input/: methotrexate_docked.mol2')
print('  From dasabuvir/input/  : dasabuvir_docked.mol2')
print('  From mdp/              : em.mdp, nvt.mdp, npt.mdp, md.mdp')
print()

uploaded = files.upload()

REQUIRED = [
    'protein_5CCV.pdb',
    'celecoxib_docked.mol2', 'methotrexate_docked.mol2', 'dasabuvir_docked.mol2',
    'em.mdp', 'nvt.mdp', 'npt.mdp', 'md.mdp',
]
missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing:
    print(f'\n*** MISSING FILES: {missing} ***')
    print('Please re-upload the missing files.')
else:
    print(f'\n=== All {len(REQUIRED)} files uploaded successfully ===')
    for f in sorted(os.listdir('.')):
        if not f.startswith('.'):
            print(f'  {f} ({os.path.getsize(f):,} bytes)')

In [ ]:
# ============================================================
# Cell 3: Verify GPU is available
# ============================================================
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n')[:10]:
        print(line)
    print('\n=== GPU detected — MD will use GPU acceleration ===')
else:
    print('*** WARNING: No GPU detected! ***')
    print('Go to Runtime > Change runtime type > T4 GPU > Save')
    print('Then re-run from Cell 1.')

In [ ]:
# ============================================================
# Cell 4: Helper functions (shared by all 3 drugs)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import os, subprocess, shutil, glob, re

WORKDIR = '/content/md_simulation'

# Standard heavy atoms for each amino acid (for detecting incomplete residues)
STANDARD_ATOMS = {
    'ALA': {'N','CA','C','O','CB'},
    'ARG': {'N','CA','C','O','CB','CG','CD','NE','CZ','NH1','NH2'},
    'ASN': {'N','CA','C','O','CB','CG','OD1','ND2'},
    'ASP': {'N','CA','C','O','CB','CG','OD1','OD2'},
    'CYS': {'N','CA','C','O','CB','SG'},
    'GLN': {'N','CA','C','O','CB','CG','CD','OE1','NE2'},
    'GLU': {'N','CA','C','O','CB','CG','CD','OE1','OE2'},
    'GLY': {'N','CA','C','O'},
    'HIS': {'N','CA','C','O','CB','CG','ND1','CD2','CE1','NE2'},
    'ILE': {'N','CA','C','O','CB','CG1','CG2','CD1'},
    'LEU': {'N','CA','C','O','CB','CG','CD1','CD2'},
    'LYS': {'N','CA','C','O','CB','CG','CD','CE','NZ'},
    'MET': {'N','CA','C','O','CB','CG','SD','CE'},
    'PHE': {'N','CA','C','O','CB','CG','CD1','CD2','CE1','CE2','CZ'},
    'PRO': {'N','CA','C','O','CB','CG','CD'},
    'SER': {'N','CA','C','O','CB','OG'},
    'THR': {'N','CA','C','O','CB','OG1','CG2'},
    'TRP': {'N','CA','C','O','CB','CG','CD1','CD2','NE1','CE2','CE3','CZ2','CZ3','CH2'},
    'TYR': {'N','CA','C','O','CB','CG','CD1','CD2','CE1','CE2','CZ','OH'},
    'VAL': {'N','CA','C','O','CB','CG1','CG2'},
}
BACKBONE = {'N', 'CA', 'C', 'O'}


def run_cmd(cmd, label='', check=True):
    """Run a shell command, print output on failure."""
    if label:
        print(f'  [{label}]')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f'  STDOUT: {result.stdout[-1000:]}')
        print(f'  STDERR: {result.stderr[-3000:]}')
        raise RuntimeError(f'Command failed: {cmd}')
    return result


def check_dependencies():
    """Verify all required tools are available before starting."""
    print('Checking dependencies...')
    ok = True
    # Check all CLI tools (acpype is a CLI tool, not a Python import)
    for tool in ['gmx', 'antechamber', 'tleap', 'parmchk2', 'acpype']:
        r = subprocess.run(f'which {tool}', shell=True,
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'  {tool}: OK ({r.stdout.strip()})')
        else:
            print(f'  {tool}: *** NOT FOUND ***')
            ok = False
    if not ok:
        raise RuntimeError(
            'Missing dependencies! Re-run Cell 1 to install them.\n'
            'If Cell 1 already ran, check its output for errors.')
    print('  All dependencies OK.\n')


def preprocess_mol2(drug_name):
    """Fix mol2 file for ACPYPE compatibility.

    Issues fixed:
    1. Molecule name "=" -> drug name (avoids antechamber filename issues)
    2. Multiple substructure IDs -> unified to single residue "LIG"
       (methotrexate has UNK0 + GLU1 split — ACPYPE needs single residue)
    3. Validate atom/bond counts
    """
    src = os.path.join(WORKDIR, f'{drug_name}_docked.mol2')
    dst = os.path.join(os.getcwd(), f'{drug_name}_fixed.mol2')

    with open(src) as f:
        content = f.read()

    lines = content.split('\n')
    out = []
    in_atoms = False
    in_section = None

    for i, line in enumerate(lines):
        # Fix molecule name (line after @<TRIPOS>MOLECULE)
        if i > 0 and lines[i-1].strip() == '@<TRIPOS>MOLECULE':
            out.append(drug_name.upper())
            continue

        # Track sections
        if line.startswith('@<TRIPOS>'):
            in_section = line.strip()
            in_atoms = (in_section == '@<TRIPOS>ATOM')
            out.append(line)
            continue

        # Fix atom lines: unify substructure ID and residue name
        if in_atoms and line.strip() and not line.startswith('@'):
            parts = line.split()
            if len(parts) >= 9:
                # parts: id, name, x, y, z, type, substruct_id,
                #         substruct_name, charge
                parts[6] = '1'        # substruct_id -> 1
                parts[7] = 'LIG'      # substruct_name -> LIG
                # Reconstruct with proper spacing
                out.append(
                    f'{int(parts[0]):>7d} {parts[1]:<4s}'
                    f'{float(parts[2]):>13.4f}'
                    f'{float(parts[3]):>10.4f}'
                    f'{float(parts[4]):>10.4f}'
                    f' {parts[5]:<8s}'
                    f'{int(parts[6]):>3d}  {parts[7]:<4s}'
                    f'    {float(parts[8]):>8.4f}'
                )
                continue

        out.append(line)

    with open(dst, 'w') as f:
        f.write('\n'.join(out))

    # Verify
    n_atoms = sum(1 for l in out
                  if not l.startswith('@') and not l.startswith('#')
                  and l.strip() and in_atoms)

    print(f'  Preprocessed {drug_name} mol2:')
    print(f'    Source: {os.path.basename(src)}')
    print(f'    Fixed:  {os.path.basename(dst)}')
    print(f'    Molecule name: {drug_name.upper()}')
    print(f'    Residue name: LIG (unified)')
    return dst


def plot_xvg(xvg_path, title, xlabel, ylabel, save_path):
    """Parse GROMACS .xvg file and plot."""
    x, y = [], []
    with open(xvg_path) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                x.append(float(parts[0]))
                y.append(float(parts[1]))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, y, linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    print(f'  Saved: {save_path}')


def fix_pdb_for_gromacs(pdb_in, pdb_out):
    """Prepare PDB for GROMACS pdb2gmx (pure Python, no dependencies).

    Handles three issues with 5CCV Chain A:
    1. Fix incomplete residues: residues with missing sidechain heavy
       atoms are mutated to ALA (keeps N,CA,C,O,CB) or GLY (keeps
       N,CA,C,O). e.g. ARG 890 missing CG -> becomes ALA 890.
    2. Rename HIS -> HIE (epsilon-protonated) to avoid interactive
       protonation prompts (24 HIS residues).
    3. Detect chain breaks (gaps in residue numbering from missing
       loops) and insert TER records. 5CCV Chain A has 3 breaks:
       404->420, 453->478, 794->801 (45 missing residues total).

    Used with pdb2gmx flags: -merge all -ter
    """
    # ---- Read PDB and group by residue ----
    with open(pdb_in) as f:
        lines = f.readlines()

    # Parse ATOM/HETATM lines, group by residue
    from collections import OrderedDict
    residues = OrderedDict()  # (resnum_str) -> {name, atoms, lines}

    for line in lines:
        if line.startswith('HETATM'):
            line = 'ATOM  ' + line[6:]
        if not line.startswith('ATOM'):
            continue
        atom_name = line[12:16].strip()
        res_name = line[17:20].strip()
        res_num = line[22:26].strip()

        if res_num not in residues:
            residues[res_num] = {
                'name': res_name,
                'atoms': set(),
                'lines': []
            }
        residues[res_num]['atoms'].add(atom_name)
        residues[res_num]['lines'].append(line)

    # ---- Step 1: Fix incomplete residues ----
    print('  Checking for incomplete residues...')
    fixed_lines = []
    n_mutated = 0
    n_removed = 0

    for res_num, res in residues.items():
        res_name = res['name']
        atoms = {a for a in res['atoms'] if not a.startswith('H')}

        if res_name not in STANDARD_ATOMS:
            # Non-standard residue — keep as-is
            fixed_lines.extend(res['lines'])
            continue

        expected = STANDARD_ATOMS[res_name]
        missing = expected - atoms

        if not missing:
            # Complete residue — keep as-is
            fixed_lines.extend(res['lines'])
            continue

        # Has missing atoms — check backbone
        missing_bb = BACKBONE - atoms
        if missing_bb:
            # Missing backbone — remove entirely
            print(f'    REMOVED {res_name} {res_num}: '
                  f'missing backbone {missing_bb}')
            n_removed += 1
            continue

        # Sidechain atoms missing — mutate to ALA or GLY
        if 'CB' in atoms:
            new_name = 'ALA'
            keep = BACKBONE | {'CB'}
        else:
            new_name = 'GLY'
            keep = BACKBONE

        print(f'    MUTATED {res_name} {res_num} -> {new_name}: '
              f'missing {missing}')
        n_mutated += 1

        for line in res['lines']:
            atom_name = line[12:16].strip()
            # Skip hydrogen atoms and atoms not in the new residue
            if atom_name.startswith('H'):
                continue
            if atom_name in keep:
                # Rename residue
                line = line[:17] + f'{new_name:>3s}' + line[20:]
                fixed_lines.append(line)

    if n_mutated == 0 and n_removed == 0:
        print('    No incomplete residues found.')
    else:
        print(f'    Fixed: {n_mutated} mutated, {n_removed} removed')

    # ---- Step 2: HIS->HIE rename ----
    out_lines = []
    for line in fixed_lines:
        if line[17:20] == 'HIS':
            line = line[:17] + 'HIE' + line[20:]
        out_lines.append(line)

    # ---- Step 3: Detect chain breaks and insert TER records ----
    out = []
    prev_resnum = None
    breaks = []
    for line in out_lines:
        resnum = int(line[22:26].strip())
        if resnum != prev_resnum:
            if prev_resnum is not None and resnum > prev_resnum + 1:
                out.append('TER\n')
                breaks.append((prev_resnum, resnum,
                               resnum - prev_resnum - 1))
            prev_resnum = resnum
        out.append(line)

    out.append('TER\n')
    out.append('END\n')

    with open(pdb_out, 'w') as f:
        f.writelines(out)

    n_his = sum(1 for l in out
                if l.startswith('ATOM') and l[17:20] == 'HIE')
    n_atoms = sum(1 for l in out if l.startswith('ATOM'))
    print(f'  PDB cleaned: {n_atoms} atoms, {n_his} HIS->HIE renamed')
    if breaks:
        print(f'  Chain breaks detected ({len(breaks)}):')
        for prev_res, next_res, gap in breaks:
            print(f'    Residue {prev_res} -> {next_res} '
                  f'({gap} missing)')
        print(f'  TER records inserted at each break for -merge all')
    else:
        print('  No chain breaks detected.')
    return pdb_out


def prepare_system(drug_name):
    """Prepare GROMACS system for one drug.

    Steps: fix PDB -> pdb2gmx -> preprocess mol2 -> ACPYPE ligand
           -> combine -> solvate -> ions -> index
    """
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.makedirs(drug_dir, exist_ok=True)
    os.chdir(drug_dir)

    protein_pdb = os.path.join(WORKDIR, 'protein_5CCV.pdb')

    print(f'\n{"="*60}')
    print(f'  PREPARING: {drug_name.upper()}')
    print(f'{"="*60}')

    # --- Step 1: Fix PDB for GROMACS ---
    print('\n[1/7] Cleaning protein PDB for GROMACS...')
    fixed_pdb = fix_pdb_for_gromacs(protein_pdb, 'protein_fixed.pdb')

    # --- Step 2: Process protein with pdb2gmx (AMBER99SB-ILDN) ---
    print('\n[2/7] Processing protein with pdb2gmx...')
    run_cmd(
        f'gmx pdb2gmx -f {fixed_pdb} -o protein.gro '
        f'-p topol.top -ignh -ff amber99sb-ildn -water tip3p '
        f'-merge all -ter',
        'pdb2gmx'
    )

    # --- Step 3: Preprocess + parametrise ligand with ACPYPE ---
    print('\n[3/7] Parametrising ligand with ACPYPE (GAFF2)...')

    # Preprocess mol2: fix molecule name "=" and unify substructure IDs
    mol2_file = preprocess_mol2(drug_name)

    # Clean any previous ACPYPE output
    for d in glob.glob('*.acpype'):
        shutil.rmtree(d, ignore_errors=True)

    # Use Gasteiger charges (-c gas) for speed and compatibility.
    # All 3 drugs use the same method so the comparison is fair.
    acpype_ok = False

    # Try 1: GAFF2 with Gasteiger charges
    print('  Attempt 1: GAFF2 + Gasteiger charges...')
    r = run_cmd(
        f'acpype -i {mol2_file} -c gas -n 0 -a gaff2',
        'acpype-gaff2', check=False
    )
    if r.returncode == 0 and glob.glob('*.acpype'):
        acpype_ok = True
        print('  ACPYPE (gaff2): SUCCESS')
    else:
        if r.returncode != 0:
            print(f'  ACPYPE gaff2 failed (exit code {r.returncode})')
            print(f'  stderr: {r.stderr[-500:]}')
        for d in glob.glob('*.acpype'):
            shutil.rmtree(d, ignore_errors=True)

        # Try 2: GAFF with Gasteiger charges
        print('  Attempt 2: GAFF + Gasteiger charges...')
        r = run_cmd(
            f'acpype -i {mol2_file} -c gas -n 0 -a gaff',
            'acpype-gaff', check=False
        )
        if r.returncode == 0 and glob.glob('*.acpype'):
            acpype_ok = True
            print('  ACPYPE (gaff): SUCCESS')
        else:
            if r.returncode != 0:
                print(f'  ACPYPE gaff failed (exit code {r.returncode})')
                print(f'  stderr: {r.stderr[-500:]}')
            for d in glob.glob('*.acpype'):
                shutil.rmtree(d, ignore_errors=True)

            # Try 3: Use existing charges from mol2
            print('  Attempt 3: GAFF2 + user charges...')
            r = run_cmd(
                f'acpype -i {mol2_file} -c user -n 0 -a gaff2',
                'acpype-user', check=False
            )
            if r.returncode == 0 and glob.glob('*.acpype'):
                acpype_ok = True
                print('  ACPYPE (user charges): SUCCESS')
            else:
                if r.returncode != 0:
                    print(f'  ACPYPE user failed (exit code {r.returncode})')
                    print(f'  stderr: {r.stderr[-1000:]}')

    if not acpype_ok:
        raise RuntimeError(
            f'ACPYPE failed for {drug_name} after 3 attempts.\n'
            f'Check mol2 file: {mol2_file}\n'
            f'Try running manually: acpype -i {mol2_file} -c gas -n 0 -a gaff2'
        )

    # Find the ACPYPE output directory
    acpype_dirs = sorted(glob.glob('*.acpype'))
    acpype_dir = acpype_dirs[0]
    print(f'  ACPYPE output: {acpype_dir}')

    # Copy GROMACS topology files from ACPYPE
    lig_itp = glob.glob(os.path.join(acpype_dir, '*_GMX.itp'))
    lig_gro = glob.glob(os.path.join(acpype_dir, '*_GMX.gro'))
    if not lig_itp or not lig_gro:
        # Fallback: try any .itp/.gro that aren't position restraints
        lig_itp = [f for f in glob.glob(os.path.join(acpype_dir, '*.itp'))
                   if 'posre' not in f.lower()]
        lig_gro = glob.glob(os.path.join(acpype_dir, '*.gro'))

    if not lig_itp:
        # List all files in acpype dir for debugging
        all_files = os.listdir(acpype_dir)
        raise FileNotFoundError(
            f'No .itp file found in {acpype_dir}!\n'
            f'Files present: {all_files}'
        )
    if not lig_gro:
        all_files = os.listdir(acpype_dir)
        raise FileNotFoundError(
            f'No .gro file found in {acpype_dir}!\n'
            f'Files present: {all_files}'
        )

    shutil.copy(lig_itp[0], 'ligand.itp')
    shutil.copy(lig_gro[0], 'ligand.gro')
    print(f'  Ligand ITP: {os.path.basename(lig_itp[0])}')
    print(f'  Ligand GRO: {os.path.basename(lig_gro[0])}')

    # Copy atomtypes if present (GAFF2 custom types)
    at_files = glob.glob(os.path.join(acpype_dir, '*atomtypes*itp'))
    has_atomtypes = False
    if at_files:
        shutil.copy(at_files[0], 'ligand_atomtypes.itp')
        has_atomtypes = True

    # --- Step 4: Combine protein + ligand into complex.gro ---
    print('\n[4/7] Combining protein + ligand...')
    with open('protein.gro') as f:
        prot_lines = f.readlines()
    with open('ligand.gro') as f:
        lig_lines = f.readlines()

    prot_natoms = int(prot_lines[1].strip())
    lig_natoms = int(lig_lines[1].strip())
    total_atoms = prot_natoms + lig_natoms

    with open('complex.gro', 'w') as f:
        f.write(f'Protein-ligand complex: {drug_name}\n')
        f.write(f'{total_atoms}\n')
        for line in prot_lines[2:2+prot_natoms]:
            f.write(line)
        for line in lig_lines[2:2+lig_natoms]:
            f.write(line)
        f.write(prot_lines[-1])  # box vector
    print(f'  Combined: {prot_natoms} protein + {lig_natoms} ligand'
          f' = {total_atoms} atoms')

    # --- Step 5: Update topology to include ligand ---
    print('\n[5/7] Updating topology...')
    with open('topol.top') as f:
        top_content = f.read()

    # Get ligand moleculetype name from [ moleculetype ] section of .itp
    # IMPORTANT: This must match what GROMACS expects in [ molecules ].
    # The [ moleculetype ] name (e.g. "celecoxib_fixed") is different
    # from the residue name in [ atoms ] (e.g. "UNL" or "LIG").
    lig_moltype = None
    with open('ligand.itp') as f:
        in_moltype = False
        for line in f:
            if '[ moleculetype ]' in line:
                in_moltype = True
                continue
            if in_moltype:
                if line.strip() and not line.startswith(';'):
                    parts = line.split()
                    if parts:
                        lig_moltype = parts[0]
                    break
    if not lig_moltype:
        raise RuntimeError(
            'Could not find [ moleculetype ] name in ligand.itp!\n'
            'Check that ACPYPE produced a valid topology file.')
    print(f'  Ligand moleculetype: {lig_moltype}')

    # Insert includes after forcefield.itp
    lines = top_content.split('\n')
    new_lines = []
    ff_done = False
    for line in lines:
        new_lines.append(line)
        if 'forcefield.itp' in line and not ff_done:
            if has_atomtypes:
                new_lines.append('#include "ligand_atomtypes.itp"')
            new_lines.append('#include "ligand.itp"')
            ff_done = True

    # Add ligand to [ molecules ] using moleculetype name
    new_lines.append(f'{lig_moltype}     1')
    # Ensure trailing newline — gmx solvate needs it to append SOL
    new_lines.append('')

    with open('topol.top', 'w') as f:
        f.write('\n'.join(new_lines))

    # --- Step 6: Solvate ---
    print('\n[6/7] Box + solvate...')
    run_cmd(
        'gmx editconf -f complex.gro -o box.gro '
        '-c -d 1.2 -bt dodecahedron',
        'editconf'
    )
    run_cmd(
        'gmx solvate -cp box.gro -cs spc216.gro '
        '-o solvated.gro -p topol.top',
        'solvate'
    )

    # Verify solvation updated topology — gmx solvate should have
    # added SOL to [ molecules ]. If not, add it manually by counting
    # SOL residues in solvated.gro.
    with open('topol.top') as f:
        top_after = f.read()
    if 'SOL' not in top_after:
        print('  WARNING: gmx solvate did not add SOL to topology!')
        print('  Adding SOL manually...')
        # Count SOL atoms in solvated.gro (3 atoms per water: OW, HW1, HW2)
        n_sol = 0
        with open('solvated.gro') as f:
            for line in f:
                if len(line) >= 10 and 'SOL' in line[:10]:
                    n_sol += 1
        n_sol_mols = n_sol // 3
        # Append SOL entry to topology
        with open('topol.top', 'a') as f:
            f.write(f'SOL              {n_sol_mols}\n')
        print(f'  Added: SOL  {n_sol_mols} ({n_sol} atoms)')
    else:
        # Count SOL for display
        m = re.search(r'SOL\s+(\d+)', top_after)
        if m:
            print(f'  Solvent added: {m.group(1)} SOL molecules')

    # --- Step 7: Add ions ---
    print('\n[7/7] Adding ions to neutralise...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "em.mdp")} '
        f'-c solvated.gro -p topol.top -o ions.tpr -maxwarn 5',
        'grompp-ions'
    )
    run_cmd(
        'echo SOL | gmx genion -s ions.tpr -o system.gro '
        '-p topol.top -pname NA -nname CL -neutral',
        'genion'
    )

    # --- Create index groups ---
    # MDP files expect tc-grps = Protein_LIG Water_and_ions
    # make_ndx auto-names the merged group "Protein_Other", so we
    # post-process the ndx file to rename it to "Protein_LIG".
    print('\n  Creating index groups...')

    result = run_cmd('echo q | gmx make_ndx -f system.gro', check=False)
    ndx_out = result.stdout + result.stderr

    print('  Default groups:')
    for line in ndx_out.split('\n'):
        line = line.strip()
        if re.match(r'\d+\s+\S+', line) and ':' in line:
            print(f'    {line}')

    # Find the "Other" group (contains ligand)
    other_grp = None
    other_name = None
    for line in ndx_out.split('\n'):
        line = line.strip()
        m = re.match(r'(\d+)\s+(Other|LIG|UNK|MOL)\b', line,
                     re.IGNORECASE)
        if m:
            other_grp = m.group(1)
            other_name = m.group(2)
            print(f'  Ligand index group: {line}')
            break

    if other_grp is None:
        other_grp = '13'
        other_name = 'Other'
        print(f'  Using default ligand group: {other_grp}')

    has_water_ions = ('Water_and_ions' in ndx_out or
                      'Water_and_Ions' in ndx_out)
    if has_water_ions:
        print('  Water_and_ions group: found')
    else:
        print('  WARNING: Water_and_ions not found in defaults')

    # Create Protein_LIG group: Protein | Ligand
    ndx_cmds = f'1 | {other_grp}\\nq\\n'
    run_cmd(
        f'printf "{ndx_cmds}" | gmx make_ndx -f system.gro -o index.ndx',
        'make_ndx',
        check=False
    )

    # Rename auto-generated group to match MDP tc-grps
    if os.path.exists('index.ndx'):
        with open('index.ndx') as f:
            ndx = f.read()

        renamed = False
        for auto_name in [f'Protein_{other_name}',
                          'Protein_Other', 'Protein_UNK',
                          'Protein_MOL', 'Protein_LIG']:
            if f'[ {auto_name} ]' in ndx:
                if auto_name != 'Protein_LIG':
                    ndx = ndx.replace(f'[ {auto_name} ]',
                                      '[ Protein_LIG ]')
                    print(f'  Renamed group: {auto_name} -> Protein_LIG')
                else:
                    print('  Protein_LIG group already exists')
                renamed = True
                break

        if not renamed:
            print('  WARNING: Could not find combined group to '
                  'rename.')

        if ('[ Water_and_Ions ]' in ndx and
                '[ Water_and_ions ]' not in ndx):
            ndx = ndx.replace('[ Water_and_Ions ]',
                              '[ Water_and_ions ]')

        with open('index.ndx', 'w') as f:
            f.write(ndx)

        groups_found = re.findall(r'\[ (\S+) \]', ndx)
        for needed in ['Protein_LIG', 'Water_and_ions']:
            status = 'found' if needed in groups_found else 'MISSING'
            print(f'  {needed}: {status}')

    print(f'\n=== {drug_name.upper()} system prepared ===')
    return drug_dir


def run_equilibration(drug_name):
    """Run EM + NVT + NPT equilibration."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)

    print(f'\n{"="*60}')
    print(f'  EQUILIBRATING: {drug_name.upper()}')
    print(f'{"="*60}')

    # --- Energy Minimisation ---
    print('\n[EM] Energy minimisation...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "em.mdp")} '
        f'-c system.gro -p topol.top -o em.tpr -maxwarn 5',
        'grompp-em'
    )
    # -ntmpi 1: use single MPI rank to avoid prime-factor domain
    # decomposition errors on Colab (26 cores = 2*13).
    # OpenMP threads handle parallelism within the single rank.
    run_cmd('gmx mdrun -v -deffnm em -ntmpi 1', 'mdrun-em')

    run_cmd('echo Potential | gmx energy -f em.edr -o em_potential.xvg',
            'energy-em', check=False)
    if os.path.exists('em_potential.xvg'):
        plot_xvg('em_potential.xvg',
                 f'{drug_name} — Energy Minimisation',
                 'Step', 'Potential Energy (kJ/mol)',
                 f'em_energy_{drug_name}.png')

    # --- NVT Equilibration ---
    print('\n[NVT] NVT equilibration (100 ps, 300K)...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "nvt.mdp")} '
        f'-c em.gro -r em.gro -p topol.top -o nvt.tpr '
        f'-n index.ndx -maxwarn 5',
        'grompp-nvt'
    )
    run_cmd('gmx mdrun -deffnm nvt -ntmpi 1', 'mdrun-nvt')

    run_cmd('echo Temperature | gmx energy -f nvt.edr -o nvt_temp.xvg',
            'energy-nvt', check=False)
    if os.path.exists('nvt_temp.xvg'):
        plot_xvg('nvt_temp.xvg',
                 f'{drug_name} — NVT Temperature',
                 'Time (ps)', 'Temperature (K)',
                 f'nvt_temp_{drug_name}.png')

    # --- NPT Equilibration ---
    print('\n[NPT] NPT equilibration (100 ps, 300K, 1 bar)...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "npt.mdp")} '
        f'-c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top '
        f'-o npt.tpr -n index.ndx -maxwarn 5',
        'grompp-npt'
    )
    run_cmd('gmx mdrun -deffnm npt -ntmpi 1', 'mdrun-npt')

    run_cmd('echo Density | gmx energy -f npt.edr -o npt_density.xvg',
            'energy-npt-density', check=False)
    if os.path.exists('npt_density.xvg'):
        plot_xvg('npt_density.xvg',
                 f'{drug_name} — NPT Density',
                 'Time (ps)', u'Density (kg/m\u00b3)',
                 f'npt_density_{drug_name}.png')

    print(f'\n=== {drug_name.upper()} equilibration complete ===')


def run_production_full(drug_name):
    """Full 50 ns production in one shot."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)

    print(f'\n{"="*60}')
    print(f'  PRODUCTION (50 ns): {drug_name.upper()}')
    print(f'{"="*60}')

    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "md.mdp")} '
        f'-c npt.gro -t npt.cpt -p topol.top -o md.tpr '
        f'-n index.ndx -maxwarn 5',
        'grompp-md'
    )
    print('  Starting 50 ns production run...')
    run_cmd('gmx mdrun -deffnm md -v -ntmpi 1', 'mdrun-production')
    print(f'\n=== {drug_name.upper()} production complete ===')


def run_production_chunked(drug_name, chunk_ns=10, total_ns=50):
    """Production in chunks for Colab Free tier."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)

    n_chunks = total_ns // chunk_ns
    steps_per_chunk = int(chunk_ns * 1e6 / 2)  # dt=0.002 ps

    print(f'\n{"="*60}')
    print(f'  PRODUCTION (CHUNKED {n_chunks}x{chunk_ns}ns): '
          f'{drug_name.upper()}')
    print(f'{"="*60}')

    for chunk in range(n_chunks):
        print(f'\n--- Chunk {chunk+1}/{n_chunks} '
              f'({chunk*chunk_ns}-{(chunk+1)*chunk_ns} ns) ---')

        if chunk == 0:
            with open(os.path.join(WORKDIR, 'md.mdp')) as f:
                mdp = f.read()
            mdp = mdp.replace('nsteps              = 25000000',
                              f'nsteps              = {steps_per_chunk}')
            with open('md_chunk.mdp', 'w') as f:
                f.write(mdp)
            run_cmd(
                f'gmx grompp -f md_chunk.mdp '
                f'-c npt.gro -t npt.cpt -p topol.top '
                f'-o md.tpr -n index.ndx -maxwarn 5',
                f'grompp-chunk{chunk+1}'
            )
        else:
            run_cmd(
                f'gmx convert-tpr -s md.tpr '
                f'-extend {chunk_ns * 1000} -o md.tpr',
                f'extend-chunk{chunk+1}'
            )

        print(f'  Running chunk {chunk+1}...')
        run_cmd(
            'gmx mdrun -deffnm md -v -cpi md.cpt -ntmpi 1',
            f'mdrun-chunk{chunk+1}'
        )
        print(f'  Chunk {chunk+1} complete!')

        try:
            dd = f'/content/drive/MyDrive/GeneTropica_MD/{drug_name}'
            os.makedirs(dd, exist_ok=True)
            for fn in ['md.cpt', 'md.xtc', 'md.edr', 'md.log']:
                if os.path.exists(fn):
                    shutil.copy(fn, dd)
            print(f'  Checkpoint saved to Drive: {dd}')
        except Exception as e:
            print(f'  (Drive save skipped: {e})')

    print(f'\n=== {drug_name.upper()} chunked production complete ===')


def package_results(drug_name):
    """Package results into tar.gz for download."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)

    result_files = [
        'md.xtc', 'md.tpr', 'md.gro', 'md.edr', 'topol.top',
        'md.cpt', 'md.log', 'index.ndx',
        'em.gro', 'npt.gro', 'system.gro', 'ligand.itp',
    ]
    result_files += glob.glob('*.png')
    if os.path.exists('ligand_atomtypes.itp'):
        result_files.append('ligand_atomtypes.itp')

    existing = [f for f in result_files if os.path.exists(f)]
    tar_name = f'md_results_{drug_name}.tar.gz'
    tar_path = os.path.join(WORKDIR, tar_name)

    run_cmd(f'tar -czf {tar_path} {" ".join(existing)}',
            f'package-{drug_name}')

    size_mb = os.path.getsize(tar_path) / 1e6
    print(f'  Packaged: {tar_name} ({size_mb:.1f} MB, '
          f'{len(existing)} files)')
    return tar_path


print('=== Helper functions loaded ===')

In [ ]:
# ============================================================
# Cell 5: (Optional) Mount Google Drive for backup
# ============================================================
# Uncomment to enable automatic checkpoint saves to Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# print('Drive mounted. Checkpoints save to /content/drive/MyDrive/GeneTropica_MD/')

In [ ]:
# ============================================================
# Cell 6: Prepare all 3 drug systems (~15 min total)
# ============================================================
os.chdir(WORKDIR)

# Pre-flight check: verify all tools are installed
check_dependencies()

# Change this list to run fewer drugs at a time, e.g.:
#   DRUGS = ['celecoxib']
DRUGS = ['celecoxib', 'methotrexate', 'dasabuvir']

for drug in DRUGS:
    prepare_system(drug)

print('\n' + '='*60)
print('  ALL SYSTEMS PREPARED SUCCESSFULLY')
print('='*60)

In [ ]:
# ============================================================
# Cell 7: Equilibrate all 3 systems (EM + NVT + NPT) (~90 min)
# ============================================================
# Verify plots after each drug:
#   EM energy  : should drop steeply then flatten
#   NVT temp   : should oscillate around 300 K
#   NPT density: should stabilise near 1000 kg/m3

for drug in DRUGS:
    run_equilibration(drug)

print('\n' + '='*60)
print('  ALL 3 SYSTEMS EQUILIBRATED')
print('  Check the plots above before proceeding!')
print('='*60)

In [ ]:
# ============================================================
# Cell 8: CHOOSE YOUR PRODUCTION MODE
# ============================================================
#
# OPTION A: 'full'    — 50 ns in one shot (Colab Pro)
# OPTION B: 'chunked' — 5 x 10 ns each (Colab Free, RECOMMENDED)

PRODUCTION_MODE = 'chunked'  # 'full' or 'chunked'

print(f'Production mode: {PRODUCTION_MODE}')
if PRODUCTION_MODE == 'chunked':
    print('Each drug: 5 chunks of 10 ns. Download between drugs.')
else:
    print('Full 50 ns per drug. Ensure stable connection.')

In [ ]:
# ============================================================
# Cell 9: CELECOXIB — Production MD
# ============================================================
# Consensus rank #1 (0.6846, ML 0.7026)
# Mechanism: COX-2 selective inhibitor
# Question: Does the pipeline's best prediction actually bind NS5 RdRp?

if PRODUCTION_MODE == 'full':
    run_production_full('celecoxib')
else:
    run_production_chunked('celecoxib', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 10: Package and download CELECOXIB results
# ============================================================
tar_path = package_results('celecoxib')

try:
    files.download(tar_path)
    print('Download started for md_results_celecoxib.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual: Files sidebar > {tar_path}')

try:
    dd = '/content/drive/MyDrive/GeneTropica_MD/'
    os.makedirs(dd, exist_ok=True)
    shutil.copy(tar_path, dd)
    print(f'Saved to Drive: {dd}')
except:
    pass

print('\n>>> Download celecoxib results before running methotrexate! <<<')

In [ ]:
# ============================================================
# Cell 11: METHOTREXATE — Production MD
# ============================================================
# Consensus rank #3 (0.4930, ML 0.4834)
# Mechanism: DHFR / host-directed
# Question: Can a host-directed drug also bind the polymerase directly?

if PRODUCTION_MODE == 'full':
    run_production_full('methotrexate')
else:
    run_production_chunked('methotrexate', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 12: Package and download METHOTREXATE results
# ============================================================
tar_path = package_results('methotrexate')

try:
    files.download(tar_path)
    print('Download started for md_results_methotrexate.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual: Files sidebar > {tar_path}')

try:
    dd = '/content/drive/MyDrive/GeneTropica_MD/'
    os.makedirs(dd, exist_ok=True)
    shutil.copy(tar_path, dd)
    print(f'Saved to Drive: {dd}')
except:
    pass

print('\n>>> Download methotrexate results before running dasabuvir! <<<')

In [ ]:
# ============================================================
# Cell 13: DASABUVIR — Production MD
# ============================================================
# Consensus rank #17 (0.3758)
# Mechanism: Non-nucleoside RdRp inhibitor (PMID 37632595)
# Question: Does cross-family RdRp conservation enable HCV->dengue binding?

if PRODUCTION_MODE == 'full':
    run_production_full('dasabuvir')
else:
    run_production_chunked('dasabuvir', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 14: Package and download DASABUVIR results
# ============================================================
tar_path = package_results('dasabuvir')

try:
    files.download(tar_path)
    print('Download started for md_results_dasabuvir.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual: Files sidebar > {tar_path}')

try:
    dd = '/content/drive/MyDrive/GeneTropica_MD/'
    os.makedirs(dd, exist_ok=True)
    shutil.copy(tar_path, dd)
    print(f'Saved to Drive: {dd}')
except:
    pass

print('\n=== ALL THREE SIMULATIONS COMPLETE ===')
print('You should now have 3 tar.gz files. Extract and run Part 2.')

## Disconnection Recovery

If Colab disconnects during production:

1. Re-run **Cell 1** (reinstalls all dependencies, ~5-8 min)
2. Re-run **Cell 2** (re-upload 8 files)
3. Re-run **Cell 3** (GPU check)
4. Re-run **Cell 4** (helper functions)
5. **Skip** Cells 6-7 if checkpoint exists
6. Jump to the drug cell where disconnection occurred
7. Chunked mode resumes from last checkpoint automatically

Use **Cell 16** below to check which checkpoints exist.

In [ ]:
# ============================================================
# Cell 16: Recovery — check checkpoint status
# ============================================================
for drug in ['celecoxib', 'methotrexate', 'dasabuvir']:
    dd = os.path.join(WORKDIR, drug)
    cpt = os.path.join(dd, 'md.cpt')
    npt = os.path.join(dd, 'npt.gro')
    if os.path.exists(cpt):
        print(f'{drug}: Production checkpoint found. Resume from production cell.')
    elif os.path.exists(npt):
        print(f'{drug}: Equilibration done. Start production cell.')
    else:
        print(f'{drug}: No checkpoint. Re-run Cell 6 (prepare) + Cell 7 (equilibrate).')